# l2p : LLM-driven Planning Model library kit

[![GitHub repo](https://img.shields.io/badge/github-repo-green)](https://github.com/AI-Planning/l2p)
[![PyPI](https://img.shields.io/pypi/v/l2p.svg)](https://pypi.org/project/l2p/)
[![License](https://img.shields.io/badge/license-MIT-blue)](https://github.com/AI-Planning/l2p/blob/main/LICENSE)
<!-- [![Tests](https://github.com/simonw/llm/workflows/Test/badge.svg)](httpsß://github.com/simonw/llm/actions?query=workflow%3ATest) -->
<!-- [![Changelog](https://img.shields.io/github/v/release/simonw/llm?include_prereleases&label=changelog)](https://llm.datasette.io/en/stable/changelog.html) -->
<!-- [![Discord](https://img.shields.io/discord/823971286308356157?label=discord)](https://datasette.io/discord-llm)
[![Homebrew](https://img.shields.io/homebrew/installs/dy/llm?color=yellow&label=homebrew&logo=homebrew)](https://formulae.brew.sh/formula/llm) -->

This library is a collection of tools for PDDL model generation extracted from natural language driven by large language models. This library is an expansion from the survey paper [**LLMs as Planning Formalizers: A Survey for Leveraging Large Language Models to Construct Automated Planning Specifications**](https://aclanthology.org/2025.findings-acl.1291.pdf).

L2P is an offline, natural language-to-planning system (that wraps an LLM backend) to support domain-agnostic planning. It does this via creating an intermediate [PDDL](https://planning.wiki/guide/whatis/pddl) representation of the domain and task, which can then be solved by a classical planner. To stay up to date with the most current papers, please visit [**here**](https://ai-planning.github.io/l2p/docs/paper_feed.html).

Full library documentation can be found: [**L2P Documention**](https://ai-planning.github.io/l2p/docs/)

# Environment Setup

Run the cells below to create a Python virtual environment and install the required dependencies for this tutorial.


## 1. Create a Virtual Environment

It is recommended to use a virtual environment to isolate dependencies. The cell below creates one called `.venv` in the current directory.


In [1]:
# Create a virtual environment
!python3 -m venv .venv
print("Virtual environment .venv created.")

Virtual environment .venv created.


> **Note:** In a notebook, the kernel itself must be set to the virtual environment. Select the `.venv` kernel from the kernel picker (top-right) before proceeding. If it does not appear, refresh the page or run:
> ```
> !python3 -m ipykernel install --user --name=.venv
> ```


## 2. Install Dependencies

Install all required packages for the tutorial.


In [ ]:
# Upgrade pip and install core dependencies
!python3 -m pip install --upgrade pip -q
%pip install l2p
%pip install llm openai
# %llm install llm-anthropic llm-deepseek

# OTHER DEPENDENCIES
%pip install requests
%pip install python-dotenv

print("Core packages installed.")

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
  Using cached llm-0.31-py3-none-any.whl.metadata (29 kB)
  Using cached openai-2.43.0-py3-none-any.whl.metadata (32 kB)
  Using cached condense_json-0.1.3-py3-none-any.whl.metadata (4.4 kB)
  Using cached click_default_group-1.2.4-py2.py3-none-any.whl.metadata (2.8 kB)
  Using cached sqlite_utils-3.39-py3-none-any.whl.metadata (7.7 kB)
  Using cached sqlite_migrate-0.1b0-py3-none-any.whl.metadata (5.4 kB)
  Using cached pluggy-1.6.0-py3-none-any.whl.metadata (4.8 kB)
  Using cached python_ulid-3.1.0-py3-none-any.whl.metadata (5.8 kB)
  Using cached setuptools-82.0.1-py3-none-any.whl.metadata (6.5 kB)
  Using cached puremagic-2.2.0-py3-none-any.whl.metadata (7.3 kB)
  Using cached anyio-4.14.0-py3-none-any.whl.metadata (4.6 kB)
  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Us

## 3. Verify Installation


In [67]:
# Verify that l2p is importable
try:
    import l2p
    print(f"l2p version: {l2p.__version__}")
except ImportError:
    print("l2p not found. Check the installation above.")
except AttributeError:
    print("l2p installed (no __version__ attribute).")

l2p installed (no __version__ attribute).


---



## 4. Setup Ollama

In [4]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Stopping running Ollama instance...
>>> Removing existing Ollama installation...
>>> Downloading Ollama for macOS...
######################################################################## 100.0%                                               11.2%                                    47.6%       77.7%
>>> Installing Ollama to /Applications...
>>> Starting Ollama...
>>> Install complete. You can now run 'ollama'.


In [ ]:
# or windows: https://ollama.com/download/windows
!irm https://ollama.com/install.ps1 | iex

In [2]:
!ollama pull gemma4:31b-cloud

]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest 
pulling 5eb7ea60f4a3: 100% ▕██████████████████▏  342 B                         
verifying sha256 digest 
writing manifest 
success 


In [3]:
!ollama list

]11;?\NAME                     ID              SIZE      MODIFIED      
gemma4:31b-cloud         c382fbfbc73b    -         6 seconds ago    
llama2:7b                78e26419b446    3.8 GB    4 weeks ago      
deepseek-v4-pro:cloud    22bfd5026abd    -         7 weeks ago      
gpt-oss:20b-cloud        875e8e3a629a    -         8 weeks ago      
ministral-3:3b-cloud     6938c17dead4    -         8 weeks ago      


## Setting up your LLM

> **Before running the cell below**, create a `.env` file in the root directory
> Then edit `.env` to set your actual key (e.g., `OLLAMA_API_KEY=sk-...`).
> 
> For example:
> ```
> OLLAMA_API_KEY="sk-your-key-here"
> OPENAI_API_KEY="sk-your-key-here"
> ANTHROPIC_API_KEY="sk-your-key-here"
> GEMINI_API_KEY="sk-your-key-here"
> DEEPSEEK_API_KEY="sk-your-key-here"
> GLM_API_KEY="sk-your-key-here"
> ...
> ```

### Connect your LLM

In [5]:
# Connect your LLM -- Simple response check!
 
# Import dependencies
import os
from dotenv import load_dotenv
from l2p import UnifiedLLM, OPENAI

In [ ]:

# load API key from .env file (see .env.example)
load_dotenv()
api_key = os.getenv("")

# Instantiate your model -- there are two options:

# 1. LLM CLASS
llm = UnifiedLLM(
    provider="",        # LLM provider (e.g., openai, google, anthropic, deepseek, etc.) 
    model="",           # LLM engine api name (e.g., gpt-5-nano, deepseek-reasoner, etc.)
    config_path="",     # Configuration file to set model arguments (e.g., cost, parameters, context length)
    api_key=api_key     # API key -- can input actual key string (not recommended!)
)

# Check if your model works!
response = llm.query(prompt="")

print(response)
print(llm.get_query_log()) # option information

[INFO] connecting to gpt-5-nano (Prompt estimation: 3 tokens)...
Hello! Nice to meet you. How can I help today? If you’re into programming, I can show you a quick “Hello, World!” in different languages, help with a project, explain concepts, or chat about anything else.

Here are a few quick snippets:
- Python: print('Hello, World!')
- JavaScript: console.log('Hello, World!')
- Java: public class Hello { public static void main(String[] args) { System.out.println("Hello, World!"); } }
- C: #include <stdio.h> int main() { printf("Hello, World!\n"); return 0; }

Want me to tailor to a specific language or project?
[{'model': 'gpt-5-nano', 'prompt': 'Hello world!', 'response': 'Hello! Nice to meet you. How can I help today? If you’re into programming, I can show you a quick “Hello, World!” in different languages, help with a project, explain concepts, or chat about anything else.\n\nHere are a few quick snippets:\n- Python: print(\'Hello, World!\')\n- JavaScript: console.log(\'Hello, Worl

In [68]:
# load API key from .env file (see .env.example)
load_dotenv()
api_key = os.getenv("OLLAMA_API_KEY") # "sk-finkksv..."

# 2. OPENAI SDK CLASS
llm = OPENAI(
    provider="ollama-cloud",
    model="gemma4:31b-cloud",
    config_path="ollama-cloud.yaml",
    api_key=api_key
)

# Check if your model works!
response = llm.query(prompt="Hello world!")

print(response)
print(llm.get_query_log()) # option information

Requesting 8192 tokens (estimated prompt: 3 tokens, margin: 200, window: 256000)
[INFO] connecting to gemma4:31b-cloud (8192 tokens)...
Hello world! How can I help you today?
[{'model': 'gemma4:31b-cloud', 'messages': [{'role': 'user', 'content': 'Hello world!'}], 'response': 'Hello world! How can I help you today?', 'input_tokens': 16, 'output_tokens': 11, 'reasoning_tokens': 0, 'total_tokens': 27, 'input_cost_usd': 0.0, 'output_cost_usd': 0.0, 'total_cost_usd': 0.0}]


---

# Section 3: DomainBuilder

The DomainBuilder constructs PDDL domain specifications. We'll use it in two ways:
first with **LLM-powered generation** from natural language, then with **manual assembly**
to see the underlying structure.


## 3a. LLM-Powered Domain Generation

We describe the classic **Blocks World** in natural language and let the LLM generate
the PDDL components for us.


In [8]:
# Import domain-building components
from l2p import DomainBuilder
from l2p.utils.pddl_types import PDDLType, Predicate, Action

# instantiate model
db = DomainBuilder()
db.set_domain_name("blocksworld")

In [9]:
# LLM: Generate types
description = """
In the blocks world, blocks sit on a table or are stacked on top of other blocks.
A robotic arm can pick up a block, put it down on the table, stack it onto another
block, or unstack it from another block. The arm can hold at most one block at a time
and can only pick up a block that has nothing on top of it.
"""

# call LLM to generate component
results, raw = db.formalize_component(
    model=llm,
    component_class=PDDLType, # <-- IMPORTANT
    description=description,
)

types = results.get(PDDLType, [])
db.set_types(types)
print(f"Generated {len(types)} type(s):", [t.name for t in types])

Requesting 8192 tokens (estimated prompt: 442 tokens, margin: 200, window: 256000)
[INFO] connecting to gemma4:31b-cloud (8192 tokens)...
Generated 1 type(s): ['block']


In [10]:
# LLM: Generate predicates (pass types as context)
results, raw = db.formalize_component(
    model=llm,
    component_class=Predicate,
    description=description,
    types=types, # <-- PASS TYPES AS CONTEXT TO FOLLOW
)

predicates = results.get(Predicate, [])
db.set_predicates(predicates)
print(f"Generated {len(predicates)} predicate(s):", [p.name for p in predicates])

Requesting 8192 tokens (estimated prompt: 560 tokens, margin: 200, window: 256000)
[INFO] connecting to gemma4:31b-cloud (8192 tokens)...
Generated 5 predicate(s): ['on', 'on-table', 'clear', 'holding', 'arm-empty']


In [11]:
# LLM: Generate actions (pass types and predicates as context)
results, raw = db.formalize_component(
    model=llm,
    component_class=Action,
    description=description,
    types=types,
    predicates=predicates, # <-- PASS AS CONTEXT
)

actions = results.get(Action, [])
db.set_actions(actions)
print(f"Generated {len(actions)} action(s):")

for a in actions:
    print(f" - {a.name}: {a.desc}")

Requesting 8192 tokens (estimated prompt: 1680 tokens, margin: 200, window: 256000)
[INFO] connecting to gemma4:31b-cloud (8192 tokens)...
Generated 4 action(s):
 - pick-up: Pick up a block from the table.
 - put-down: Put a block down on the table.
 - stack: Stack a block onto another block.
 - unstack: Unstack a block from another block.


In [12]:
# Auto-generate requirements and assemble the domain PDDL
requirements = db.generate_requirements(db.domain_details)
db.set_requirements(requirements)
print("Requirements:", [r.name for r in requirements])


Requirements: [':equality', ':negative-preconditions', ':strips', ':typing']


In [13]:
# Generate the full domain PDDL string
domain_pddl = db.generate_domain(db.domain_details)
print(domain_pddl)

(define (domain blocksworld)
   (:requirements
      :equality :negative-preconditions :strips :typing)

   (:types 
      block - object
   )

   (:predicates 
      (arm-empty )
      (clear ?b - block)
      (holding ?b - block)
      (on ?b1 - block ?b2 - block)
      (on-table ?b - block)
   )

   (:action pick-up
     :parameters (?b - block)
     :precondition (and (arm-empty) (on-table ?b) (clear ?b))
     :effect (and (holding ?b) (not (arm-empty)) (not (on-table ?b)) (not (clear ?b)))
   )
   
   (:action put-down
     :parameters (?b - block)
     :precondition (holding ?b)
     :effect (and (arm-empty) (on-table ?b) (clear ?b) (not (holding ?b)))
   )
   
   (:action stack
     :parameters (?b ?x - block)
     :precondition (and (holding ?b) (clear ?x) (not (= ?b ?x)))
     :effect (and (arm-empty) (on ?b ?x) (not (holding ?b)) (not (clear ?x)))
   )
   
   (:action unstack
     :parameters (?b ?x - block)
     :precondition (and (arm-empty) (on ?b ?x) (clear ?b))
     :eff

In [27]:
# Write the LLM-generated domain to disk
with open("blocksworld_llm.pddl", "w") as f:
    f.write(domain_pddl)
print("Written to blocksworld_llm.pddl")


Written to blocksworld_llm.pddl


## 3b. Manual Domain Assembly

Now we build the same Blocks World domain by hand to understand exactly what the LLM generated.
We construct Pydantic models directly and feed them into the builder.


In [14]:
from l2p import Parameter, ActionPrecondition, ActionEffect

# Define types
types_manual = [PDDLType(name="block", parent="object")]

# Define predicates
predicates_manual = [
    Predicate(name="on", params=[
        Parameter(variable="?x", type="block"),
        Parameter(variable="?y", type="block"),
    ]),
    Predicate(name="ontable", params=[Parameter(variable="?x", type="block")]),
    Predicate(name="clear", params=[Parameter(variable="?x", type="block")]),
    Predicate(name="holding", params=[Parameter(variable="?x", type="block")]),
    Predicate(name="handempty", params=[]),
]


In [15]:
# Define actions manually
actions_manual = [
    Action(
        name="pick-up",
        params=[Parameter(variable="?x", type="block")],
        preconditions=ActionPrecondition(conditions=[
            "(clear ?x)", "(ontable ?x)", "(handempty)"
        ]),
        effects=ActionEffect(
            add=["(holding ?x)"],
            delete=["(clear ?x)", "(ontable ?x)", "(handempty)"]
        ),
    ),
    Action(
        name="put-down",
        params=[Parameter(variable="?x", type="block")],
        preconditions=ActionPrecondition(conditions=["(holding ?x)"]),
        effects=ActionEffect(
            add=["(ontable ?x)", "(clear ?x)", "(handempty)"],
            delete=["(holding ?x)"]
        ),
    ),
    Action(
        name="stack",
        params=[
            Parameter(variable="?x", type="block"),
            Parameter(variable="?y", type="block"),
        ],
        preconditions=ActionPrecondition(conditions=[
            "(holding ?x)", "(clear ?y)"
        ]),
        effects=ActionEffect(
            add=["(on ?x ?y)", "(clear ?x)", "(handempty)"],
            delete=["(holding ?x)", "(clear ?y)"]
        ),
    ),
    Action(
        name="unstack",
        params=[
            Parameter(variable="?x", type="block"),
            Parameter(variable="?y", type="block"),
        ],
        preconditions=ActionPrecondition(conditions=[
            "(on ?x ?y)", "(clear ?x)", "(handempty)"
        ]),
        effects=ActionEffect(
            add=["(holding ?x)", "(clear ?y)"],
            delete=["(on ?x ?y)", "(clear ?x)", "(handempty)"]
        ),
    ),
]

In [16]:
# Assemble the manual domain
db_manual = DomainBuilder()
db_manual.set_domain_name("blocksworld")
db_manual.set_types(types_manual)
db_manual.set_predicates(predicates_manual)
db_manual.set_actions(actions_manual)

reqs_manual = db_manual.generate_requirements(db_manual.domain_details)
db_manual.set_requirements(reqs_manual)

domain_pddl_manual = db_manual.generate_domain(db_manual.domain_details)
print(domain_pddl_manual)

(define (domain blocksworld)
   (:requirements
      :strips :typing)

   (:types 
      block - object
   )

   (:predicates 
      (clear ?x - block)
      (handempty )
      (holding ?x - block)
      (on ?x - block ?y - block)
      (ontable ?x - block)
   )

   (:action pick-up
     :parameters (?x - block)
     :precondition (and (clear ?x) (ontable ?x) (handempty))
     :effect (and (holding ?x) (not (clear ?x)) (not (ontable ?x)) (not (handempty)))
   )
   
   (:action put-down
     :parameters (?x - block)
     :precondition (holding ?x)
     :effect (and (ontable ?x) (clear ?x) (handempty) (not (holding ?x)))
   )
   
   (:action stack
     :parameters (?x ?y - block)
     :precondition (and (holding ?x) (clear ?y))
     :effect (and (on ?x ?y) (clear ?x) (handempty) (not (holding ?x)) (not (clear ?y)))
   )
   
   (:action unstack
     :parameters (?x ?y - block)
     :precondition (and (on ?x ?y) (clear ?x) (handempty))
     :effect (and (holding ?x) (clear ?y) (not (on ?x 

In [35]:
# Write the manually-built domain to disk
with open("blocksworld.pddl", "w") as f:
    f.write(domain_pddl_manual)
print("Written to blocksworld.pddl")


Written to blocksworld.pddl


## 3c. Domain Validation

We validate both domains using `DomainValidator` to check for structural errors.


In [17]:
from l2p import DomainValidator

dv = DomainValidator()

# Validate the LLM-generated domain
result_llm = dv.validate_domain(db.domain_details)

print("LLM domain valid:", result_llm.valid)
if not result_llm.valid:
    for e in result_llm.errors:
        print(f"  ERROR: {e}")
    for w in result_llm.warnings:
        print(f"  WARN: {w}")
print(vars(result_llm))
print()

LLM domain valid: True
{'valid': True, 'errors': [], 'warnings': []}



If we want to just validate against a single component:

In [18]:
result = dv.validate_component(
    target=db.domain_details.actions[0], # <-- retrieve first action 
    context={PDDLType: types, Predicate: predicates} # <-- list context as dictionary <key=l2p.type> <value=list[l2p.type]>
)

print(vars(result))

{'valid': True, 'errors': [], 'warnings': []}


In [19]:
for a in db.domain_details.actions: 
    result = dv.validate_component(
        target=a,
        context={PDDLType: types, Predicate: predicates}
    )
    print(f"Checking action: {a.name}")
    print(f"Validation result: {vars(result)}")

Checking action: pick-up
Validation result: {'valid': True, 'errors': [], 'warnings': []}
Checking action: put-down
Validation result: {'valid': True, 'errors': [], 'warnings': []}
Checking action: stack
Validation result: {'valid': True, 'errors': [], 'warnings': []}
Checking action: unstack
Validation result: {'valid': True, 'errors': [], 'warnings': []}


In [20]:
# Validate the manually-built domain
result_manual = dv.validate_domain(db_manual.domain_details)
print("Manual domain valid:", result_manual.valid)
if not result_manual.valid:
    for e in result_manual.errors:
        print(f"  ERROR: {e}")
    for w in result_manual.warnings:
        print(f"  WARN: {w}")
print(vars(result_manual))

Manual domain valid: True
{'valid': True, 'errors': [], 'warnings': []}


---

# Section 4: ProblemBuilder

The ProblemBuilder constructs PDDL problem specifications (objects, initial state, goal).
We again do it both with LLM assistance and manually.


## 4a. LLM-Powered Problem Generation

We describe a specific Blocks World scenario and let the LLM generate the problem components.


In [21]:
from l2p import ProblemBuilder, ProblemDetails

# NL description of the problem scenario
problem_desc = """
Given the blocksworld domain where blocks sit on a table, create a problem with
three blocks named a, b, and c. Initially all three blocks are on the table and clear.
The robot arm is empty. The goal is to have block a on top of b, and block b on top of c.
"""

In [22]:
# LLM: Generate the full problem at once
results, raw = ProblemBuilder().formalize_component(
    model=llm,
    component_class=ProblemDetails,
    description=problem_desc,
    types=db.domain_details.types,
    predicates=db.domain_details.predicates
)

problem         = results[ProblemDetails][0] # <-- extract problem details
objects         = problem.objects
initial_state   = problem.initial_state
goal_state      = problem.goal_state

print(f"Objects: {[o.name for o in objects]}")
print(f"Initial facts: {initial_state.facts if initial_state else None}")
print(f"Goal conditions: {goal_state.conditions if goal_state else None}")

Requesting 8192 tokens (estimated prompt: 1710 tokens, margin: 200, window: 256000)
[INFO] connecting to gemma4:31b-cloud (8192 tokens)...
Objects: ['a', 'b', 'c']
Initial facts: ['(on-table a)', '(on-table b)', '(on-table c)', '(clear a)', '(clear b)', '(clear c)', '(arm-empty)']
Goal conditions: ['(on a b)', '(on b c)']


In [23]:
# Assemble the LLM-generated problem
pb_llm = ProblemBuilder(
    name="blocksworld-pb1",
    domain_name="blocksworld",
    objects=objects,
    initial_state=initial_state if initial_state else None,
    goal_state=goal_state if goal_state else None,
)

problem_pddl_llm = pb_llm.generate_problem(pb_llm.problem_details)
print(problem_pddl_llm)

(define (problem blocksworld-pb1)
   (:domain blocksworld)

   (:objects 
      a b c - block
   )

   (:init 
      (on-table a)
      (on-table b)
      (on-table c)
      (clear a)
      (clear b)
      (clear c)
      (arm-empty)
   )

   (:goal 
      (and (on a b) (on b c))
   )
)


In [ ]:
# Write the LLM-generated problem to disk
with open("blocksworld_problem_llm.pddl", "w") as f:
    f.write(problem_pddl_llm)
print("Written to blocksworld_problem_llm.pddl")

Written to blocksworld_problem_llm.pddl


## 4b. Manual Problem Assembly

Building the same problem manually to see the exact structure.


In [24]:
from l2p import PDDLObject, InitialState, GoalState

pb_manual = ProblemBuilder(
    name="blocksworld-pb1",
    domain_name="blocksworld",
    objects=[
        PDDLObject(name="a", type="block"),
        PDDLObject(name="b", type="block"),
        PDDLObject(name="c", type="block"),
    ],
    initial_state=InitialState(facts=[
        "(ontable a)", "(ontable b)", "(ontable c)",
        "(clear a)", "(clear b)", "(clear c)",
        "(handempty)",
    ]),
    goal_state=GoalState(conditions=[
        "(on a b)", "(on b c)"
    ]),
)

problem_pddl_manual = pb_manual.generate_problem(pb_manual.problem_details)
print(problem_pddl_manual)

(define (problem blocksworld-pb1)
   (:domain blocksworld)

   (:objects 
      a b c - block
   )

   (:init 
      (ontable a)
      (ontable b)
      (ontable c)
      (clear a)
      (clear b)
      (clear c)
      (handempty)
   )

   (:goal 
      (and (on a b) (on b c))
   )
)


In [34]:
# Write the manually-built problem to disk
with open("blocksworld_problem.pddl", "w") as f:
    f.write(problem_pddl_manual)
print("Written to blocksworld_problem.pddl")

Written to blocksworld_problem.pddl


## 4c. Problem Validation

Validate both problems using `ProblemValidator`.


In [27]:
from l2p import ProblemValidator

pv = ProblemValidator()

# Validate the LLM-generated problem
res_prob_llm = pv.validate_problem(pb_llm.problem_details, domain=db.domain_details)
print("LLM problem valid:", res_prob_llm.valid)
if not res_prob_llm.valid:
    for e in res_prob_llm.errors:
        print(f"  ERROR: {e}")
print(vars(res_prob_llm))
print()

LLM problem valid: True
{'valid': True, 'errors': [], 'warnings': []}



In [28]:
pv.validate_component(
    target=pb_llm.problem_details.objects,
    context={PDDLType: db.domain_details.types}
)

print(vars(result))

{'valid': True, 'errors': [], 'warnings': []}


In [29]:
# Validate the manually-built problem
res_prob_manual = pv.validate_problem(pb_manual.problem_details, domain=db_manual.domain_details)
print("Manual problem valid:", res_prob_manual.valid)
if not res_prob_manual.valid:
    for e in res_prob_manual.errors:
        print(f"  ERROR: {e}")
print(vars(res_prob_manual))

Manual problem valid: True
{'valid': True, 'errors': [], 'warnings': []}


---

# Section 5: PlannerBuilder

Now we run the **FastDownward** planner on our domain and problem to find a valid plan.
FastDownward is a classical AI planner that solves PDDL planning tasks.


In [36]:
from l2p.planner_builder import FastDownward

# Instantiate the planner with the path to the FastDownward executable
planner = FastDownward(executable_path="downward/fast-downward.py")

# Run the planner on our manually-built domain and problem
result = planner.run_planner(
    domain_path="blocksworld.pddl",
    problem_path="blocksworld_problem.pddl",
    timeout=30,
)

print("Planner successful:", result.is_successful)
print("Error:", result.error_message)

Planner successful: True
Error: None


In [37]:
# FastDownward writes the plan to a file called 'sas_plan'
with open("sas_plan", "r") as f:
    plan_steps = f.read()

print("=== Found Plan ===")
print(plan_steps)


=== Found Plan ===
(pick-up b)
(stack b c)
(pick-up a)
(stack a b)
; cost = 4 (unit cost)



---

# Section 6: FeedbackBuilder

The FeedbackBuilder provides LLM-driven diagnosis and repair of faulty PDDL code.
We'll introduce deliberate errors into our domain, let the validator catch them,
then use `llm_diagnose` and `llm_revise` to fix them.


In [30]:
# Build a faulty domain: missing the 'holding' predicate
db_faulty = DomainBuilder()
db_faulty.set_domain_name("blocksworld-faulty")
db_faulty.set_types([PDDLType(name="block", parent="object")])
db_faulty.set_predicates([
    Predicate(name="on", params=[
        Parameter(variable="?x", type="block"),
        Parameter(variable="?y", type="block"),
    ]),
    Predicate(name="ontable", params=[Parameter(variable="?x", type="block")]),
    Predicate(name="clear", params=[Parameter(variable="?x", type="block")]),
    Predicate(name="handempty", params=[]),
    # NOTE: "holding" predicate is deliberately missing!
])
db_faulty.set_actions([
    Action(
        name="pick-up",
        params=[Parameter(variable="?x", type="block")],
        preconditions=ActionPrecondition(conditions=[
            "(clear ?x)", "(ontable ?x)", "(handempty)"
        ]),
        effects=ActionEffect(
            add=["(holding ?x)"],  # NOTE: uses undeclared predicate in action!
            delete=["(clear ?x)", "(ontable ?x)", "(handempty)"]
        ),
    ),
])

reqs_faulty = db_faulty.generate_requirements(db_faulty.domain_details)
db_faulty.set_requirements(reqs_faulty)

In [31]:
# Validate the faulty domain to see the errors
result_faulty = dv.validate_domain(db_faulty.domain_details)
print("Domain valid:", result_faulty.valid)

for e in result_faulty.errors:
    print(e)
    print()

for w in result_faulty.warnings:
    print(w)
    print()

Domain valid: False
[actions] [ERROR] Action 'pick-up' effects uses undeclared predicate/function keyword 'holding'. Please ensure 'holding' is properly defined in the domain.
Allowed predicates/functions: [clear, handempty, on, ontable]

[predicates] [WARNING] Predicate 'on' is declared but never used in any action, event, or process.



In [32]:
from l2p import FeedbackBuilder

fb = FeedbackBuilder()

# Step 1: Diagnose the errors
diagnosis, raw_diag = fb.llm_diagnose(
    model=llm,
    artifact=db_faulty.domain_details,      # <-- PASS IN FAULTY COMPONENT (e.g., DomainDetails)
    errors="\n".join(result_faulty.errors), # <-- PASS IN ERROR STRINGS
    description=description,
)
print("=== Diagnosis ===")
print(diagnosis)


Requesting 8192 tokens (estimated prompt: 921 tokens, margin: 200, window: 256000)
[INFO] connecting to gemma4:31b-cloud (8192 tokens)...
=== Diagnosis ===
{
  "summary": "The domain definition is missing a required predicate 'holding' which is used as an effect in the 'pick-up' action.",
  "identified_errors": [
    {
      "error_type": "UndeclaredVariable",
      "location_in_json": "actions[0].effects.add[0]",
      "validator_message": "Action 'pick-up' effects uses undeclared predicate/function keyword 'holding'. Please ensure 'holding' is properly defined in the domain.",
      "root_cause_analysis": "The generator defined the logic for picking up a block but forgot to include the 'holding' predicate in the global predicates list."
    }
  ],
  "repair_plan": [
    "Add a new predicate object to the 'predicates' array with the name 'holding' and a single parameter of type 'block' (e.g., {'name': 'holding', 'params': [{'variable': '?x', 'type': 'block'}]})."
  ]
}


In [33]:
from l2p import DomainDetails

# Step 2: Revise the domain based on the diagnosis
revised, raw_revise = fb.llm_revise(
    model=llm,
    artifact=db_faulty.domain_details,
    component_class=DomainDetails,
    diagnosis=str(diagnosis),
    description=description,
)

fixed_domain = revised.get(DomainDetails, [None])[0]

print("=== Revised Domain ===")
if fixed_domain:
    db_fixed = DomainBuilder(domain_details=fixed_domain)
    fixed_pddl = db_fixed.generate_domain(fixed_domain)
    print(fixed_pddl)

    # Validate the fixed version
    result_fixed = dv.validate_domain(fixed_domain)
    print("Fixed domain valid:", result_fixed.valid)
else:
    print("No revised domain returned.")

Requesting 8192 tokens (estimated prompt: 880 tokens, margin: 200, window: 256000)
[INFO] connecting to gemma4:31b-cloud (8192 tokens)...
=== Revised Domain ===
(define (domain blocksworld-faulty)
   (:requirements
      :strips :typing)

   (:types 
      block - object
   )

   (:predicates 
      (clear ?x - block)
      (handempty )
      (holding ?x - block)
      (on ?x - block ?y - block)
      (ontable ?x - block)
   )

   (:action pick-up
     :parameters (?x - block)
     :precondition (and (clear ?x) (ontable ?x) (handempty))
     :effect (and (holding ?x) (not (clear ?x)) (not (ontable ?x)) (not (handempty)))
   )
)
Fixed domain valid: True


---

# Section 8: Experiments - Algorithmic Approaches to Domain Generation

In this section we compare three algorithms for generating PDDL domains from natural language.
All three operate on the **DEPOT** (modified) domain with hardcoded types and predicates; only the
action-generation strategy differs.

| Algorithm | Strategy | Paper / Inspiration |
|---|---|---|
| **1** | Full domain generated in one `formalize_component` call | Baseline one-shot generation |
| **2** | Each action generated independently (no cross-action context) | NL2PDDL-style, action-by-action |
| **3** | Action-by-action with growing context + restatement | Generate-and-repair, each call restates all actions |


## 8a. Setup - Hardcoded Domain Definitions

Types, predicates, and the domain description are defined here. Only actions will be generated by the LLM.


In [65]:
from l2p import PDDLType, Predicate, Parameter, Action, DomainBuilder, DomainValidator

# --- Types ---
DEPOT_TYPES = [
    PDDLType(name="hoist", parent="object"),
    PDDLType(name="truck", parent="object"),
    PDDLType(name="crate", parent="object"),
    PDDLType(name="surface", parent="object"),
    PDDLType(name="place", parent="object"),
]

# --- Predicates ---
DEPOT_PREDICATES = [
    Predicate(name="at-hoist", params=[
        Parameter(variable="?h", type="hoist"),
        Parameter(variable="?p", type="place"),
    ]),
    Predicate(name="at-truck", params=[
        Parameter(variable="?t", type="truck"),
        Parameter(variable="?p", type="place"),
    ]),
    Predicate(name="at-crate", params=[
        Parameter(variable="?c", type="crate"),
        Parameter(variable="?p", type="place"),
    ]),
    Predicate(name="on", params=[
        Parameter(variable="?c", type="crate"),
        Parameter(variable="?s", type="surface"),
    ]),
    Predicate(name="in-crate", params=[
        Parameter(variable="?c", type="crate"),
        Parameter(variable="?t", type="truck"),
    ]),
    Predicate(name="lifting", params=[
        Parameter(variable="?h", type="hoist"),
        Parameter(variable="?c", type="crate"),
    ]),
    Predicate(name="clear", params=[
        Parameter(variable="?s", type="surface"),
    ]),
    Predicate(name="available", params=[
        Parameter(variable="?h", type="hoist"),
    ]),
    Predicate(name="surface-at", params=[
        Parameter(variable="?s", type="surface"),
        Parameter(variable="?p", type="place"),
    ]),
]

# --- NL description ---
DEPOT_DESCRIPTION = """
The Depots domain involves moving crates between places using trucks and hoists.
A hoist at a place can lift a crate off a surface or drop a crate onto a clear surface.
A hoist can only hold one crate at a time.
A hoist can load a crate onto a truck that is at the same place, or unload a crate from a truck.
A truck can drive between different places.
A hoist can move between places at the same location.
"""

# --- Action names with one-liner descriptions ---
DEPOT_ACTIONS = [
    ("lift", "A hoist lifts a crate off a surface at the same place."),
    ("drop", "A hoist drops a crate onto a clear surface at the same place."),
    ("load", "A hoist loads a crate onto a truck at the same place."),
    ("unload", "A hoist unloads a crate from a truck at the same place."),
    ("drive", "A truck drives from one place to another."),
    ("move-hoist", "A hoist moves between two places."),
]

dv = DomainValidator()

## 8b. Algorithm 1 - One-Shot Action Set Generation

Generate **all 6 actions** in a single `formalize_component` call with `component_class=Action`.
The LLM receives all action descriptions at once and produces the complete action set in one response.


In [69]:
# Algorithm 1: Generate all actions at once
action_desc = "Generate ALL actions for the Depots domain.\n\n"
for name, desc in DEPOT_ACTIONS:
    action_desc += f"  - {name}: {desc}\n"
action_desc += f"\n{DEPOT_DESCRIPTION}"

results, raw = DomainBuilder().formalize_component(
    model=llm,
    component_class=Action,
    description=action_desc,
    types=DEPOT_TYPES,
    predicates=DEPOT_PREDICATES,
)
actions_algo1 = results.get(Action, [])
print(f"Generated {len(actions_algo1)} action(s): {[a.name for a in actions_algo1]}")

Requesting 8192 tokens (estimated prompt: 2065 tokens, margin: 200, window: 256000)
[INFO] connecting to gemma4:31b-cloud (8192 tokens)...
Generated 6 action(s): ['lift', 'drop', 'load', 'unload', 'drive', 'move-hoist']


In [70]:
# Assemble and validate Algorithm 1
db1 = DomainBuilder()
db1.set_domain_name("depots")
db1.set_types(DEPOT_TYPES)
db1.set_predicates(DEPOT_PREDICATES)
db1.set_actions(actions_algo1)
reqs1 = db1.generate_requirements(db1.domain_details)
db1.set_requirements(reqs1)
pddl1 = db1.generate_domain(db1.domain_details)

with open("depots_algo1.pddl", "w") as f:
    f.write(pddl1)
print("Written to depots_algo1.pddl")
print(pddl1)

r1 = dv.validate_domain(db1.domain_details)
print("Valid:", r1.valid)
if not r1.valid:
    for e in r1.errors:
        print(f"  ERROR: {e}")

Written to depots_algo1.pddl
(define (domain depots)
   (:requirements
      :equality :negative-preconditions :strips :typing)

   (:types 
      crate - object
      hoist - object
      place - object
      surface - object
      truck - object
   )

   (:predicates 
      (at-crate ?c - crate ?p - place)
      (at-hoist ?h - hoist ?p - place)
      (at-truck ?t - truck ?p - place)
      (available ?h - hoist)
      (clear ?s - surface)
      (in-crate ?c - crate ?t - truck)
      (lifting ?h - hoist ?c - crate)
      (on ?c - crate ?s - surface)
      (surface-at ?s - surface ?p - place)
   )

   (:action lift
     :parameters (?c - crate ?h - hoist ?p - place ?s - surface)
     :precondition (and (at-hoist ?h ?p) (at-crate ?c ?p) (on ?c ?s) (surface-at ?s ?p) (available ?h))
     :effect (and (lifting ?h ?c) (not (available ?h)) (not (on ?c ?s)))
   )
   
   (:action drop
     :parameters (?c - crate ?h - hoist ?p - place ?s - surface)
     :precondition (and (at-hoist ?h ?p) (lif

## 8c. Algorithm 2 - NL2PDDL: Independent Action-by-Action

Each action is generated independently with **no knowledge of other actions**.
Only types and predicates are passed as context. This mirrors the NL2PDDL approach
where actions are generated one-by-one in isolation.


In [ ]:
# Algorithm 2: Generate each action independently
actions_algo2 = []
for name, desc in DEPOT_ACTIONS:
    action_desc = f"Generate the '{name}' action. {desc}\n\n{DEPOT_DESCRIPTION}"
    results, _ = DomainBuilder().formalize_component(
        model=llm,
        component_class=Action,
        description=action_desc,
        types=DEPOT_TYPES,
        predicates=DEPOT_PREDICATES,
        # NOTE: no actions= passed - each call is independent!
    )
    generated = results.get(Action, [])
    actions_algo2.extend(generated)
    print(f"  {name}: {len(generated)} action(s) generated")

Requesting 8192 tokens (estimated prompt: 1975 tokens, margin: 200, window: 256000)
[INFO] connecting to gemma4:31b-cloud (8192 tokens)...
  lift: 1 action(s) generated
Requesting 8192 tokens (estimated prompt: 1976 tokens, margin: 200, window: 256000)
[INFO] connecting to gemma4:31b-cloud (8192 tokens)...
  drop: 1 action(s) generated
Requesting 8192 tokens (estimated prompt: 1975 tokens, margin: 200, window: 256000)
[INFO] connecting to gemma4:31b-cloud (8192 tokens)...
  load: 1 action(s) generated
Requesting 8192 tokens (estimated prompt: 1976 tokens, margin: 200, window: 256000)
[INFO] connecting to gemma4:31b-cloud (8192 tokens)...
  unload: 1 action(s) generated
Requesting 8192 tokens (estimated prompt: 1970 tokens, margin: 200, window: 256000)
[INFO] connecting to gemma4:31b-cloud (8192 tokens)...
  drive: 1 action(s) generated
Requesting 8192 tokens (estimated prompt: 1972 tokens, margin: 200, window: 256000)
[INFO] connecting to gemma4:31b-cloud (8192 tokens)...
  move-hoist:

In [ ]:
# Assemble and validate Algorithm 2
db2 = DomainBuilder()
db2.set_domain_name("depots")
db2.set_types(DEPOT_TYPES)
db2.set_predicates(DEPOT_PREDICATES)
db2.set_actions(actions_algo2)
reqs2 = db2.generate_requirements(db2.domain_details)
db2.set_requirements(reqs2)
pddl2 = db2.generate_domain(db2.domain_details)

with open("depots_algo2.pddl", "w") as f:
    f.write(pddl2)
print("Written to depots_algo2.pddl")
print(pddl2)

r2 = dv.validate_domain(db2.domain_details)
print("Valid:", r2.valid)
if not r2.valid:
    for e in r2.errors:
        print(f"  ERROR: {e}")

Written to depots_algo2.pddl
(define (domain depots)
   (:requirements
      :equality :negative-preconditions :strips :typing)

   (:types 
      crate - object
      hoist - object
      place - object
      surface - object
      truck - object
   )

   (:predicates 
      (at-crate ?c - crate ?p - place)
      (at-hoist ?h - hoist ?p - place)
      (at-truck ?t - truck ?p - place)
      (available ?h - hoist)
      (clear ?s - surface)
      (in-crate ?c - crate ?t - truck)
      (lifting ?h - hoist ?c - crate)
      (on ?c - crate ?s - surface)
      (surface-at ?s - surface ?p - place)
   )

   (:action lift
     :parameters (?c - crate ?h - hoist ?p - place ?s - surface)
     :precondition (and (at-hoist ?h ?p) (at-crate ?c ?p) (surface-at ?s ?p) (on ?c ?s) (available ?h))
     :effect (and (lifting ?h ?c) (clear ?s) (not (available ?h)) (not (on ?c ?s)))
   )
   
   (:action drop
     :parameters (?c - crate ?h - hoist ?p - place ?s - surface)
     :precondition (and (lifting ?

## 8d. Algorithm 3 - Action-by-Action with Growing Repair

Each new action is generated with all prior actions as context. The prompt instructs the LLM
to **restate all previously generated actions** in every response, allowing it to repair
inconsistencies as new actions are added. This uses a custom prompt template.


In [72]:
from l2p import PromptBuilder, Action

pb = PromptBuilder()

# Step 1: Role
pb.set_role("You are an expert PDDL Generator Agent. Based on the natural language description (found under `## TASK`), your role is to model PDDL domain actions (:action) in the following format.")

# Step 2: Output format with example
pb.set_format("End your final answer by wrapping the action definitions inside specific XML tag `<actions> ... </actions>` using the JSON format shown below. Do not include Markdown backticks.")
pb.set_format_example(Action, is_list=True)

# Step 3: Rules for careful extraction
pb.add_rule("The JSON block above is strictly an ILLUSTRATIVE EXAMPLE. Do not copy names like 'navigate', 'rover', or 'battery-level' unless explicitly defined in the domain description. You must extract actual actions, variables, conditions, and effects from the text.")
pb.add_rule("Strict JSON & XML Wrapping: Output strictly valid JSON wrapped in the <actions> tags. Do not include trailing commas, and do not wrap the JSON in Markdown formatting backticks (e.g., ```json ```).")
pb.add_rule("Required Action Fields: Every action object MUST have 'name', 'params', 'preconditions', 'effects', and optional 'desc'. 'name': The action name as a string. 'params': A list of parameter objects. 'preconditions': An object containing the action preconditions. 'effects': An object containing the action effects. 'desc': Optional natural language description of the action.")
pb.add_rule("Parameter Objects: The 'params' list must contain objects with 'variable' and 'type' keys. 'variable': Must be a string beginning with a question mark (e.g., ?r, ?from). 'type': Must be a valid object type for that parameter. Parameters must logically correspond to the action described in the natural language input.")
pb.add_rule("Variable Naming: All parameter variables used in 'params', 'preconditions', and 'effects' must begin with a question mark (e.g., ?r, ?x) and must match the parameters of that action.")
pb.add_rule("Preconditions Object: The 'preconditions' object must contain 'conditions' and optional 'desc'. 'conditions': A list of logical conditions. Multiple entries in 'conditions' are implicitly joined by 'and'. Use plain strings for simple predicates and numeric comparisons. Use dictionaries for structured logic such as 'not', 'and', 'or', 'imply', 'forall', and 'exists'.")
pb.add_rule("Effects Object: The 'effects' object must contain 'add', 'delete', 'numeric', 'conditional', and optional 'desc'. 'add': Positive boolean facts made true by the action. 'delete': Boolean facts removed by the action. Do not wrap them in 'not'; just list the fact itself. 'numeric': Numeric update expressions such as (increase ...), (decrease ...), (assign ...), (scale-up ...), or (scale-down ...). 'conditional': A list of conditional effects using the PDDL when structure.")
pb.add_rule("Conditional Effects: Each item in the 'conditional' list must be an object with 'condition', 'effect', and 'desc'. 'condition': A list of logical conditions that trigger the conditional effect. 'effect': An object containing 'add', 'delete', and 'numeric' lists. Use conditional effects only when the natural language description explicitly implies an effect that occurs only under certain circumstances.")
pb.add_rule("Logical Condition Format: Valid condition dictionaries include: {\"operator\": \"not\", \"condition\": \"(pred ?x)\"}, {\"operator\": \"and\", \"conditions\": [\"(pred1 ?x)\", \"(pred2 ?x)\"]}, {\"operator\": \"or\", \"conditions\": [\"(pred1 ?x)\", \"(pred2 ?x)\"]}, {\"operator\": \"imply\", \"antecedent\": [\"(pred1 ?x)\"], \"consequent\": [\"(pred2 ?x)\"]}, {\"quantifier\": \"forall\", \"parameters\": [{\"variable\": \"?x\", \"type\": \"type\"}], \"conditions\": [\"(pred ?x)\"]}, {\"quantifier\": \"exists\", \"parameters\": [{\"variable\": \"?x\", \"type\": \"type\"}], \"conditions\": [\"(pred ?x)\"]}")
pb.add_rule("Empty Arrays: If a field has no values, you must explicitly return an empty list []. Do not omit required keys from the JSON. If an action has no preconditions, use \"preconditions\": {\"conditions\": []}. If an action has no added, deleted, numeric, or conditional effects, use empty lists for those fields.")
pb.add_rule("Restate All Actions: The <existing_context> below contains all previously generated actions inside <actions>. You must restate EVERY one of them in your output, making any changes needed for consistency across the full set. If no <actions> block exists inside <existing_context> (i.e., this is the first action), only generate the requested new action. Do not omit or drop any previously generated action.")
pb.add_rule("Do NOT add a desc=[...] field for the action preconditions or effects.")

# Step 4: Task
pb.set_task("Below are the actions that have been generated so far. You must restate ALL of them in your output, making any changes needed for consistency. If the list is empty, only generate the requested new action.")

# Generate the prompt template
CUSTOM_ACTION_PROMPT = pb.generate_prompt()
print("=== Generated Prompt Template ====")
print(CUSTOM_ACTION_PROMPT)

=== Generated Prompt Template ====
## ROLE
You are an expert PDDL Generator Agent. Based on the natural language description (found under `## TASK`), your role is to model PDDL domain actions (:action) in the following format.

## OUTPUT FORMAT
End your final answer by wrapping the action definitions inside specific XML tag `<actions> ... </actions>` using the JSON format shown below. Do not include Markdown backticks.

<actions>
[
    {
        "name": "move-rover",
        "params": [
            {
                "variable": "?r",
                "type": "rover"
            },
            {
                "variable": "?from",
                "type": "waypoint"
            },
            {
                "variable": "?to",
                "type": "waypoint"
            }
        ],
        "preconditions": {
            "conditions": [
                "(at ?r ?from)",
                {
                    "operator": "not",
                    "condition": "(= ?from ?to)"
         

In [73]:
# Algorithm 3: Generate each action with restatement of all prior actions
accumulated_actions = []
for i, (name, desc) in enumerate(DEPOT_ACTIONS):
    action_desc = f"Generate the '{name}' action. {desc}\n\n{DEPOT_DESCRIPTION}"
    results, raw = DomainBuilder().formalize_component(
        model=llm,
        component_class=Action,
        description=action_desc,
        prompt_template=CUSTOM_ACTION_PROMPT,
        types=DEPOT_TYPES,
        predicates=DEPOT_PREDICATES,
        actions=accumulated_actions,  # grows each iteration
    )
    
    accumulated_actions = results.get(Action, [])
    print(f"  {name}: {len(accumulated_actions)} total action(s) after iteration {i+1}")
    print(f"    Action names: {[a.name for a in accumulated_actions]}")

Requesting 8192 tokens (estimated prompt: 2018 tokens, margin: 200, window: 256000)
[INFO] connecting to gemma4:31b-cloud (8192 tokens)...
  lift: 1 total action(s) after iteration 1
    Action names: ['lift']
Requesting 8192 tokens (estimated prompt: 2255 tokens, margin: 200, window: 256000)
[INFO] connecting to gemma4:31b-cloud (8192 tokens)...
  drop: 2 total action(s) after iteration 2
    Action names: ['lift', 'drop']
Requesting 8192 tokens (estimated prompt: 2483 tokens, margin: 200, window: 256000)
[INFO] connecting to gemma4:31b-cloud (8192 tokens)...
  load: 3 total action(s) after iteration 3
    Action names: ['lift', 'drop', 'load']
Requesting 8192 tokens (estimated prompt: 2703 tokens, margin: 200, window: 256000)
[INFO] connecting to gemma4:31b-cloud (8192 tokens)...
  unload: 4 total action(s) after iteration 4
    Action names: ['lift', 'drop', 'load', 'unload']
Requesting 8192 tokens (estimated prompt: 2925 tokens, margin: 200, window: 256000)
[INFO] connecting to gem

In [74]:
# Assemble and validate Algorithm 3
db3 = DomainBuilder()
db3.set_domain_name("depots")
db3.set_types(DEPOT_TYPES)
db3.set_predicates(DEPOT_PREDICATES)
db3.set_actions(accumulated_actions)
reqs3 = db3.generate_requirements(db3.domain_details)
db3.set_requirements(reqs3)
pddl3 = db3.generate_domain(db3.domain_details)

with open("depots_algo3.pddl", "w") as f:
    f.write(pddl3)
print("Written to depots_algo3.pddl")
print(pddl3)

r3 = dv.validate_domain(db3.domain_details)
print("Valid:", r3.valid)
if not r3.valid:
    for e in r3.errors:
        print(f"  ERROR: {e}")

Written to depots_algo3.pddl
(define (domain depots)
   (:requirements
      :conditional-effects :equality :negative-preconditions :strips :typing)

   (:types 
      crate - object
      hoist - object
      place - object
      surface - object
      truck - object
   )

   (:predicates 
      (at-crate ?c - crate ?p - place)
      (at-hoist ?h - hoist ?p - place)
      (at-truck ?t - truck ?p - place)
      (available ?h - hoist)
      (clear ?s - surface)
      (in-crate ?c - crate ?t - truck)
      (lifting ?h - hoist ?c - crate)
      (on ?c - crate ?s - surface)
      (surface-at ?s - surface ?p - place)
   )

   (:action lift
     :parameters (?c - crate ?h - hoist ?p - place ?s - surface)
     :precondition (and (at-hoist ?h ?p) (surface-at ?s ?p) (at-crate ?c ?p) (on ?c ?s) (available ?h))
     :effect (and (lifting ?h ?c) (clear ?s) (not (on ?c ?s)) (not (available ?h)))
   )
   
   (:action drop
     :parameters (?c - crate ?h - hoist ?p - place ?s - surface)
     :precond

## 8e. Summary

| Aspect | Algorithm 1 (One-Shot) | Algorithm 2 (NL2PDDL) | Algorithm 3 (Repair) |
|---|---|---|---|
| **Strategy** | All actions in one call | Each action independently | Action-by-action with restatement |
| **LLM calls** | 1 | 6 (one per action) | 6 (one per action) |
| **Context per call** | All action descriptions | Types + predicates only | Types + predicates + all prior actions |
| **Cross-action awareness** | Yes (same response) | None | Yes (growing context) |
| **Best for** | Simple domains with few actions | Fixed, independent action vocabularies | Domains with interdependent actions |


---

# Section 9: PromptBuilder - Building Multi-Component Extraction Prompts

`PromptBuilder` provides a structured API for constructing markdown prompt templates.
In this section we build a custom prompt for **extracting both types and predicates**
from a domain description, demonstrating multi-component extraction.


## 9a. PromptBuilder API Overview

`PromptBuilder` uses a fluent builder pattern - each method returns `self` for chaining:

| Method | Purpose |
|---|---|
| `set_role(str)` | Sets `## ROLE` section |
| `set_format(str)` | Sets `## OUTPUT FORMAT` section |
| `set_format_example(cls)` | Appends a JSON example for a Pydantic model class |
| `add_rule(str)` | Appends a numbered rule to `## RULES` |
| `add_example(str)` | Appends a numbered n-shot example |
| `set_task(str)` | Sets `## TASK` section |
| `generate_prompt(**kwargs)` | Returns the assembled prompt template |
| `save_prompt(filename, **kwargs)` | Saves the prompt to a markdown file |


## 9b. Building a Custom Extraction Prompt for Types and Predicates

We build a prompt that asks the LLM to **analyze a domain description** and extract the
PDDL types and predicates, with step-by-step reasoning.


In [45]:
from l2p import PromptBuilder, PDDLType, Predicate

pb = PromptBuilder()

# Step 1: Role
pb.set_role("You are an expert PDDL knowledge engineer. Your job is to analyze domain descriptions and extract the PDDL types and predicates that define the domain.")

# Step 2: Output format with examples for both components
pb.set_format("Use the following JSON schemas wrapped in XML tags.")
pb.set_format_example(PDDLType, is_list=True)
pb.set_format_example(Predicate, is_list=True)

# Step 3: Rules for careful extraction
pb.add_rule("Read the description carefully and identify all distinct object categories (these become types).")
pb.add_rule("For each type, determine its parent (use 'object' as default).")
pb.add_rule("Identify all relationships, properties, and states in the domain (these become predicates). Think about what needs to be true before and after each action.")
pb.add_rule("Each predicate must have a descriptive name and typed parameters using the types you defined.")
pb.add_rule("Output BOTH <types> and <predicates> sections. Do not omit either.")
pb.add_rule("Use only information from the description. Do not invent types or predicates not implied by the text.")

# Step 4: Task
pb.set_task("Based on the domain description below, extract all PDDL types and predicates.\n\nThink step by step: first identify what objects exist in the world and how they relate, then output both sections.")

# Generate the prompt template
prompt_template = pb.generate_prompt()
print("=== Generated Prompt Template ====")
print(prompt_template)

=== Generated Prompt Template ====
## ROLE
You are an expert PDDL knowledge engineer. Your job is to analyze domain descriptions and extract the PDDL types and predicates that define the domain.

## OUTPUT FORMAT
Use the following JSON schemas wrapped in XML tags.

<types>
[
    {
        "name": "vehicle",
        "parent": "object",
        "desc": "Optional (str)"
    }
]
</types>

<predicates>
[
    {
        "name": "at",
        "params": [
            {
                "variable": "?r",
                "type": "rover"
            },
            {
                "variable": "?w",
                "type": "waypoint"
            }
        ],
        "desc": "Optional (str)"
    }
]
</predicates>

## RULES
1. Read the description carefully and identify all distinct object categories (these become types).
2. For each type, determine its parent (use 'object' as default).
3. Identify all relationships, properties, and states in the domain (these become predicates). Think about what nee

## 9c. Saving the Prompt to a Markdown File


In [47]:
pb.save_prompt("extract_types_predicates_prompt.md")
print("Saved to extract_types_predicates_prompt.md")

[SUCCESS] Prompt saved to: /Users/marcustantakoun/Desktop/ICAPS-Tutorial/extract_types_predicates_prompt.md
Saved to extract_types_predicates_prompt.md


In [48]:
with open("extract_types_predicates_prompt.md", "r") as f:
    content = f.read()
print(content)

## ROLE
You are an expert PDDL knowledge engineer. Your job is to analyze domain descriptions and extract the PDDL types and predicates that define the domain.

## OUTPUT FORMAT
Use the following JSON schemas wrapped in XML tags.

<types>
[
    {
        "name": "vehicle",
        "parent": "object",
        "desc": "Optional (str)"
    }
]
</types>

<predicates>
[
    {
        "name": "at",
        "params": [
            {
                "variable": "?r",
                "type": "rover"
            },
            {
                "variable": "?w",
                "type": "waypoint"
            }
        ],
        "desc": "Optional (str)"
    }
]
</predicates>

## RULES
1. Read the description carefully and identify all distinct object categories (these become types).
2. For each type, determine its parent (use 'object' as default).
3. Identify all relationships, properties, and states in the domain (these become predicates). Think about what needs to be true before and after each

## 9d. Using the Custom Prompt with `formalize_component`

The prompt template handles multi-component extraction - the LLM outputs both `<types>`
and `<predicates>` tags, and `formalize_component` parses them separately.


In [49]:
with open("extract_types_predicates_prompt.md", "r") as f:
    custom_prompt = f.read()

results, raw = DomainBuilder().formalize_component(
    model=llm,
    component_class=[PDDLType, Predicate],
    description=DEPOT_DESCRIPTION,
    prompt_template=custom_prompt,
)

print("=== LLM RAW OUTPUT ===")
print(raw)

print("\n=== PARSED RESULTS ===")
extracted_types = results.get(PDDLType, [])
extracted_preds = results.get(Predicate, [])
print(f"Types ({len(extracted_types)}):", [t.name for t in extracted_types])
for t in extracted_types:
    print(f"  {t.name} < {t.parent}")
print(f"Predicates ({len(extracted_preds)}):", [p.name for p in extracted_preds])
for p in extracted_preds:
    params = ", ".join(f"{pv.variable}:{pv.type}" for pv in p.params)
    print(f"  ({p.name} {params})")

Requesting 8192 tokens (estimated prompt: 418 tokens, margin: 200, window: 256000)
[INFO] connecting to gemma4:31b-cloud (8192 tokens)...
=== LLM RAW OUTPUT ===
<types>
[
    {
        "name": "place",
        "parent": "object",
        "desc": "A location where trucks, hoists, and crates can be positioned."
    },
    {
        "name": "crate",
        "parent": "object",
        "desc": "An item to be moved."
    },
    {
        "name": "truck",
        "parent": "object",
        "desc": "A vehicle used to transport crates between places."
    },
    {
        "name": "hoist",
        "parent": "object",
        "desc": "A mechanism used to lift, drop, load, and unload crates."
    }
]
</types>

<predicates>
[
    {
        "name": "at",
        "params": [
            {
                "variable": "?obj",
                "type": "object"
            },
            {
                "variable": "?p",
                "type": "place"
            }
        ],
        "desc": "Indicat

In [118]:
print("=== Ground Truth Types ===")
print([t.name for t in DEPOT_TYPES])
print()
print("=== Ground Truth Predicates ===")
print([p.name for p in DEPOT_PREDICATES])
print()
print("=== Extracted Types ===")
print([t.name for t in extracted_types])
print()
print("=== Extracted Predicates ===")
print([p.name for p in extracted_preds])


=== Ground Truth Types ===
['hoist', 'truck', 'crate', 'surface', 'place']

=== Ground Truth Predicates ===
['at-hoist', 'at-truck', 'at-crate', 'on', 'in-crate', 'lifting', 'clear', 'available', 'surface-at']

=== Extracted Types ===
['place', 'crate', 'truck', 'hoist']

=== Extracted Predicates ===
['at', 'holding', 'on_surface', 'in_truck', 'surface_clear', 'hoist_empty']


In [50]:
# perform validation
result = dv.validate_component(
    target=extracted_preds,
    context={PDDLType: extracted_types}
)

print(vars(result))

{'valid': True, 'errors': [], 'warnings': []}


## 9e. Summary

| Aspect | Default Prompt | Custom Prompt |
|---|---|---|
| **Target** | Single component (e.g., actions) | Multi-component (types + predicates) |
| **Output format** | Single XML tag | Multiple XML tags (`<types>` + `<predicates>`) |
| **Reasoning** | Direct generation | Step-by-step extraction |


---
# END OF TUTORIAL

Now go off exploring the library! Try out your own prompts, domain problems, different models and pipelines.

Marcus Tantakoun, PhD School of Computing, Queen's University, Kingston Canada.

`Contact`: 20mt1@queensu.ca